# Coffio Model Analyzer

This notebook loads the exported Coffio history (CSV format) and validates the grind size model:

- **Linear model**: `grindSize = a * targetYield + b * brewTime + c` (per coffee + sieve)
- **Brew ratio shift**: Linear shift when brew ratio changes (e.g. 1:2 → 1:3)
- **Sieve linking**: Constant grind size offset between sieves at the same brew ratio and brew time

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded successfully.')

## 1. Load Exported History

The Coffio app exports a CSV with sections `TABLE:COFFEES`, `TABLE:SIEVES`, and `TABLE:BREWS`.
Place your exported `.csv` file in this folder and update the path below.

In [ ]:
# --- Configuration ---
EXPORT_FILE = 'coffio_export.csv'  # Update this path to your exported file

def parse_coffio_csv(filepath: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Parse the multi-table Coffio CSV export format."""
    coffees = []
    sieves = []
    brews = []
    current_table = None
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('TABLE:'):
                current_table = line.split('TABLE:')[1]
                next(f)  # skip header line (we define our own columns)
                continue
            
            parts = line.split(',')
            if current_table == 'COFFEES' and len(parts) >= 2:
                coffees.append({'id': int(parts[0]), 'name': parts[1]})
            elif current_table == 'SIEVES' and len(parts) >= 2:
                sieves.append({'id': int(parts[0]), 'name': parts[1]})
            elif current_table == 'BREWS' and len(parts) >= 12:
                brews.append({
                    'id': int(parts[0]),
                    'coffeeId': int(parts[1]),
                    'sieveId': int(parts[2]),
                    'temperature': float(parts[3]),
                    'coffeeWeight': float(parts[4]),
                    'targetYield': float(parts[5]),
                    'actualYield': float(parts[6]),
                    'tamperPressure': float(parts[7]),
                    'milkVolume': float(parts[8]),
                    'grindSize': float(parts[9]),
                    'brewTime': int(parts[10]),
                    'timestamp': int(parts[11])
                })
    
    df_coffees = pd.DataFrame(coffees)
    df_sieves = pd.DataFrame(sieves)
    df_brews = pd.DataFrame(brews)
    
    return df_coffees, df_sieves, df_brews

# Parse the export
df_coffees, df_sieves, df_brews = parse_coffio_csv(EXPORT_FILE)

# Merge names for readability
df = df_brews.merge(df_coffees.rename(columns={'id': 'coffeeId', 'name': 'coffeeName'}), on='coffeeId', how='left')
df = df.merge(df_sieves.rename(columns={'id': 'sieveId', 'name': 'sieveName'}), on='sieveId', how='left')

# Compute brew ratio
df['brewRatio'] = df['actualYield'] / df['coffeeWeight']
df['targetRatio'] = df['targetYield'] / df['coffeeWeight']
df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')

# Filter valid brews
df_valid = df[(df['grindSize'] > 0) & (df['brewTime'] > 0)].copy()

print(f'Loaded: {len(df_coffees)} coffees, {len(df_sieves)} sieves, {len(df_brews)} brews')
print(f'Valid brews (grindSize > 0 & brewTime > 0): {len(df_valid)}')
df_valid.head()

## 2. Model Check: Grind Size vs Brew Time (per Sieve)

For each coffee+sieve combination, we expect a linear relationship between grind size and brew time
at a fixed brew ratio.

In [ ]:
# Plot grind size vs brew time, colored by sieve, for each coffee
coffees_with_data = df_valid['coffeeName'].unique()

for coffee_name in coffees_with_data:
    coffee_df = df_valid[df_valid['coffeeName'] == coffee_name]
    sieves_in_coffee = coffee_df['sieveName'].unique()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(sieves_in_coffee)))
    
    for i, sieve_name in enumerate(sieves_in_coffee):
        sieve_df = coffee_df[coffee_df['sieveName'] == sieve_name]
        ax.scatter(sieve_df['brewTime'], sieve_df['grindSize'], 
                   color=colors[i], label=f'{sieve_name}', alpha=0.7, s=60)
        
        # Fit linear regression for this sieve
        if len(sieve_df) >= 2:
            X = sieve_df[['targetYield', 'brewTime']].values
            y = sieve_df['grindSize'].values
            model = LinearRegression().fit(X, y)
            
            # Plot regression line at 1:2 ratio
            avg_coffee_weight = sieve_df['coffeeWeight'].mean()
            target_yield_1_2 = avg_coffee_weight * 2.0
            
            time_range = np.linspace(sieve_df['brewTime'].min(), sieve_df['brewTime'].max(), 50)
            X_pred = np.column_stack([np.full_like(time_range, target_yield_1_2), time_range])
            y_pred = model.predict(X_pred)
            
            ax.plot(time_range, y_pred, color=colors[i], linestyle='--', linewidth=2,
                    label=f'{sieve_name} fit (1:2)')
    
    ax.set_xlabel('Brew Time (seconds)')
    ax.set_ylabel('Grind Size')
    ax.set_title(f'Grind Size vs Brew Time — {coffee_name}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Model Check: Brew Ratio Effect

For the same coffee + sieve, changing the brew ratio (e.g. 1:2 → 1:3) should produce a linear
shift in the required brew time. We verify this by plotting targetYield vs grindSize.

In [ ]:
# Plot grind size vs target yield (brew ratio proxy), at similar brew times
for coffee_name in coffees_with_data:
    coffee_df = df_valid[df_valid['coffeeName'] == coffee_name]
    sieves_in_coffee = coffee_df['sieveName'].unique()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(sieves_in_coffee)))
    
    for i, sieve_name in enumerate(sieves_in_coffee):
        sieve_df = coffee_df[coffee_df['sieveName'] == sieve_name]
        ax.scatter(sieve_df['targetRatio'], sieve_df['grindSize'],
                   color=colors[i], label=f'{sieve_name}', alpha=0.7, s=60)
        
        # Fit and show trend
        if len(sieve_df) >= 2:
            X = sieve_df[['targetRatio']].values
            y = sieve_df['grindSize'].values
            model = LinearRegression().fit(X, y)
            ratio_range = np.linspace(sieve_df['targetRatio'].min(), sieve_df['targetRatio'].max(), 50)
            y_pred = model.predict(ratio_range.reshape(-1, 1))
            ax.plot(ratio_range, y_pred, color=colors[i], linestyle='--', linewidth=2)
    
    ax.set_xlabel('Target Brew Ratio (targetYield / coffeeWeight)')
    ax.set_ylabel('Grind Size')
    ax.set_title(f'Grind Size vs Brew Ratio — {coffee_name}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Model Check: Sieve Linking (Constant Offset)

For the same coffee, brew ratio, and brew time, the grind size difference between sieves
should be approximately constant. We verify this by:
1. Fitting a shared-slope model across all sieves
2. Computing per-sieve intercept offsets
3. Checking that the residuals are consistent

In [ ]:
# Sieve linking analysis: fit shared slopes, compute per-sieve offsets
print('='*60)
print('SIEVE LINKING ANALYSIS')
print('='*60)

for coffee_name in coffees_with_data:
    coffee_df = df_valid[df_valid['coffeeName'] == coffee_name].copy()
    sieves_in_coffee = coffee_df['sieveName'].unique()
    
    if len(sieves_in_coffee) < 2:
        print(f'\n{coffee_name}: Only 1 sieve, skipping offset analysis.')
        continue
    
    print(f'\n--- {coffee_name} ({len(sieves_in_coffee)} sieves) ---')
    
    # Create dummy variables for sieves (one-hot, drop first as reference)
    sieve_dummies = pd.get_dummies(coffee_df['sieveName'], prefix='sieve', drop_first=False)
    reference_sieve = sieves_in_coffee[0]
    
    # Fit model: grindSize = a*targetYield + b*brewTime + sum(offset_i * sieve_i)
    X = pd.concat([
        coffee_df[['targetYield', 'brewTime']].reset_index(drop=True),
        sieve_dummies.reset_index(drop=True)
    ], axis=1)
    y = coffee_df['grindSize'].values
    
    model = LinearRegression(fit_intercept=True).fit(X, y)
    
    print(f'  Global slope (targetYield): {model.coef_[0]:.4f}')
    print(f'  Global slope (brewTime):    {model.coef_[1]:.4f}')
    print(f'  R² score:                   {model.score(X, y):.4f}')
    print(f'  Intercept:                  {model.intercept_:.4f}')
    print(f'  Sieve offsets:')
    
    sieve_coefs = dict(zip(sieve_dummies.columns, model.coef_[2:]))
    ref_offset = sieve_coefs.get(f'sieve_{reference_sieve}', 0)
    for sieve_col, coef in sieve_coefs.items():
        sieve_name = sieve_col.replace('sieve_', '')
        print(f'    {sieve_name}: {coef - ref_offset:+.2f} (relative to {reference_sieve})')

In [ ]:
# Visualize sieve offsets: plot residuals after removing shared trend
for coffee_name in coffees_with_data:
    coffee_df = df_valid[df_valid['coffeeName'] == coffee_name].copy()
    sieves_in_coffee = coffee_df['sieveName'].unique()
    
    if len(sieves_in_coffee) < 2:
        continue
    
    # Fit model without sieve dummies (shared slope only)
    X_shared = coffee_df[['targetYield', 'brewTime']].values
    y = coffee_df['grindSize'].values
    shared_model = LinearRegression().fit(X_shared, y)
    coffee_df['residual'] = y - shared_model.predict(X_shared)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot of residuals per sieve
    sns.boxplot(data=coffee_df, x='sieveName', y='residual', ax=axes[0])
    axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[0].set_title(f'Residual Distribution per Sieve — {coffee_name}')
    axes[0].set_xlabel('Sieve')
    axes[0].set_ylabel('Residual (grindSize - shared prediction)')
    
    # Parallel regression lines showing the constant offset
    colors = plt.cm.tab10(np.linspace(0, 1, len(sieves_in_coffee)))
    for i, sieve_name in enumerate(sieves_in_coffee):
        sieve_df = coffee_df[coffee_df['sieveName'] == sieve_name]
        axes[1].scatter(sieve_df['brewTime'], sieve_df['grindSize'],
                        color=colors[i], label=sieve_name, alpha=0.7, s=50)
        
        # Plot the linked model line (shared slopes + sieve offset)
        if len(sieve_df) >= 2:
            offset = sieve_df['residual'].mean()
            avg_target_yield = sieve_df['targetYield'].mean()
            time_range = np.linspace(coffee_df['brewTime'].min(), coffee_df['brewTime'].max(), 50)
            X_line = np.column_stack([np.full_like(time_range, avg_target_yield), time_range])
            y_line = shared_model.predict(X_line) + offset
            axes[1].plot(time_range, y_line, color=colors[i], linestyle='--', linewidth=2)
    
    axes[1].set_xlabel('Brew Time (seconds)')
    axes[1].set_ylabel('Grind Size')
    axes[1].set_title(f'Sieve-Linked Model (Parallel Lines) — {coffee_name}')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## 5. Full Model Validation

Fit the complete model with all three components and check prediction accuracy:
- `grindSize = a * targetYield + b * brewTime + c + sieveOffset[sieve]`

In [ ]:
# Full model validation with R², MAE, and prediction vs actual plots
from sklearn.metrics import mean_absolute_error, r2_score

for coffee_name in coffees_with_data:
    coffee_df = df_valid[df_valid['coffeeName'] == coffee_name].copy()
    sieves_in_coffee = coffee_df['sieveName'].unique()
    
    print(f'\n{"="*60}')
    print(f'FULL MODEL: {coffee_name}')
    print(f'{"="*60}')
    
    # Build feature matrix with sieve indicators
    sieve_dummies = pd.get_dummies(coffee_df['sieveName'], prefix='sieve', drop_first=True)
    X = pd.concat([
        coffee_df[['targetYield', 'brewTime']].reset_index(drop=True),
        sieve_dummies.reset_index(drop=True)
    ], axis=1)
    y = coffee_df['grindSize'].values
    
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)
    
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    
    print(f'  R² score:  {r2:.4f}')
    print(f'  MAE:       {mae:.4f}')
    print(f'  Coefficients:')
    print(f'    targetYield: {model.coef_[0]:.4f}')
    print(f'    brewTime:    {model.coef_[1]:.4f}')
    print(f'    intercept:   {model.intercept_:.4f}')
    for col, coef in zip(sieve_dummies.columns, model.coef_[2:]):
        print(f'    {col}: {coef:+.4f}')
    
    # Prediction vs Actual plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].scatter(y, y_pred, alpha=0.7, s=50)
    min_val = min(y.min(), y_pred.min())
    max_val = max(y.max(), y_pred.max())
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect fit')
    axes[0].set_xlabel('Actual Grind Size')
    axes[0].set_ylabel('Predicted Grind Size')
    axes[0].set_title(f'Predicted vs Actual — {coffee_name} (R²={r2:.3f})')
    axes[0].legend()
    
    # Residuals over time
    residuals = y - y_pred
    axes[1].scatter(coffee_df['datetime'].values, residuals, alpha=0.7, s=50)
    axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Residual')
    axes[1].set_title(f'Residuals Over Time — {coffee_name}')
    
    plt.tight_layout()
    plt.show()

## 6. Sieve Offset Summary

Summary table of constant grind size shifts between sieves.

In [ ]:
# Create a summary table of sieve offsets per coffee
offset_records = []

for coffee_name in coffees_with_data:
    coffee_df = df_valid[df_valid['coffeeName'] == coffee_name].copy()
    sieves_in_coffee = sorted(coffee_df['sieveName'].unique())
    
    if len(sieves_in_coffee) < 2:
        continue
    
    # Fit shared model to get offsets
    X_shared = coffee_df[['targetYield', 'brewTime']].values
    y = coffee_df['grindSize'].values
    shared_model = LinearRegression().fit(X_shared, y)
    coffee_df['residual'] = y - shared_model.predict(X_shared)
    
    # Compute mean offset per sieve
    sieve_means = coffee_df.groupby('sieveName')['residual'].mean()
    reference = sieves_in_coffee[0]
    
    for sieve in sieves_in_coffee:
        offset_records.append({
            'Coffee': coffee_name,
            'Sieve': sieve,
            'Offset (vs ' + reference + ')': sieve_means[sieve] - sieve_means[reference],
            'N brews': len(coffee_df[coffee_df['sieveName'] == sieve])
        })

if offset_records:
    df_offsets = pd.DataFrame(offset_records)
    print('\nSieve Offset Summary:')
    print(df_offsets.to_string(index=False))
else:
    print('Not enough data with multiple sieves to compute offsets.')

## 7. 3D Visualization: Grind Size = f(Brew Time, Brew Ratio)

Interactive view showing how grind size depends on both brew time and brew ratio simultaneously.

In [ ]:
# 3D scatter: grindSize vs brewTime vs targetRatio, colored by sieve
from mpl_toolkits.mplot3d import Axes3D

for coffee_name in coffees_with_data:
    coffee_df = df_valid[df_valid['coffeeName'] == coffee_name].copy()
    sieves_in_coffee = coffee_df['sieveName'].unique()
    
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(sieves_in_coffee)))
    
    for i, sieve_name in enumerate(sieves_in_coffee):
        sieve_df = coffee_df[coffee_df['sieveName'] == sieve_name]
        ax.scatter(sieve_df['brewTime'], sieve_df['targetRatio'], sieve_df['grindSize'],
                   color=colors[i], label=sieve_name, alpha=0.7, s=50)
    
    ax.set_xlabel('Brew Time (s)')
    ax.set_ylabel('Target Ratio')
    ax.set_zlabel('Grind Size')
    ax.set_title(f'3D Model View — {coffee_name}')
    ax.legend()
    plt.tight_layout()
    plt.show()